<a href="https://colab.research.google.com/github/idialloaka-ai/DAILYCHALLENGE/blob/master/ExercisesXP_Diabetes_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Exercises XP - Diabetes Classification

## What you will learn
- Understanding the problem
- Data collection
- Model training for classification
- Model evaluation

## What you will create
- A Logistic Regression model to predict diabetes



## Exercise 1 - Understanding the problem and Data Collection

We want to predict if an individual has diabetes.

- Load the diabetes dataset and explore it
- Count positive and negative cases
- Split the data into train and test


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── Chargement du dataset ──────────────────────────────────────────────────────
df = pd.read_csv('diabetes_prediction_dataset.csv')

print("Shape du dataset :", df.shape)
print()
print("Aperçu des 5 premières lignes :")
df.head()

In [ ]:
# ── Exploration générale ───────────────────────────────────────────────────────
print("Types de données :")
print(df.dtypes)
print()
print("Valeurs manquantes :", df.isnull().sum().sum())
print()
print("Statistiques descriptives :")
df.describe()

In [ ]:
# ── Vérification de la colonne cible ──────────────────────────────────────────
assert 'diabetes' in df.columns, "Expected a 'diabetes' target column"

counts = df['diabetes'].value_counts()
print("Distribution de la variable cible 'diabetes' :")
print(f"  Cas négatifs (0 – pas de diabète) : {counts[0]:>6}  ({counts[0]/len(df)*100:.1f}%)")
print(f"  Cas positifs (1 – diabète)        : {counts[1]:>6}  ({counts[1]/len(df)*100:.1f}%)")

# Visualisation de la distribution
plt.figure(figsize=(5, 3))
plt.bar(['Pas de diabète (0)', 'Diabète (1)'], [counts[0], counts[1]],
        color=['steelblue', 'tomato'])
plt.title("Distribution des classes")
plt.ylabel("Nombre d'individus")
for i, v in enumerate([counts[0], counts[1]]):
    plt.text(i, v + 500, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Train / Test Split ────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split

X = df.drop(columns=['diabetes'])
y = df['diabetes']

# 80 % entraînement, 20 % test — stratifié pour conserver la proportion des classes
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Taille X_train :", X_train.shape)
print("Taille X_test  :", X_test.shape)
print()


## Exercise 2 - Model picking and standardization

- Which model can we use and why
- Do we need to standardize
- If yes, apply StandardScaler



> TODO: In a short paragraph, justify Logistic Regression for this binary task. Mention linear decision boundary, calibrated probabilities, and interpretability. Explain why standardization helps for numerical stability and better conditioning.




**Quel modèle utiliser et pourquoi ?**

La **régression logistique** est le choix naturel pour ce problème de classification binaire (diabète : oui/non) pour trois raisons principales :

1. **Frontière de décision linéaire** : Elle modélise log-odds comme une combinaison linéaire des features. Avec des features bien préparées (HbA1c, glycémie, âge, IMC), une séparation linéaire est souvent suffisante pour obtenir de bonnes performances.
2. **Probabilités calibrées** : Elle fournit directement `predict_proba`, permettant d'ajuster le seuil de décision selon le contexte médical (on peut vouloir maximiser le rappel pour ne pas manquer un diabétique).
3. **Interprétabilité** : Les coefficients indiquent directement l'importance et le sens de chaque variable — essentiel en médecine pour valider le modèle auprès des cliniciens.

**Faut-il standardiser ?**

Oui. La régression logistique utilise une descente de gradient qui converge plus vite et de façon plus stable quand les features sont à la même échelle. Par exemple, `blood_glucose_level` (~80–300) et `hypertension` (0 ou 1) ont des plages très différentes. Sans standardisation, le solveur peut nécessiter beaucoup plus d'itérations, voire mal converger. On applique donc `StandardScaler` sur les variables numériques et `OneHotEncoder` sur les variables catégorielles.

In [ ]:
# ── Pipeline de pré-traitement ────────────────────────────────────────────────
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = X.select_dtypes(include=['object', 'str']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

preprocess = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)
])

print("Colonnes catégorielles :", cat_cols)
print("Colonnes numériques    :", num_cols)

## Exercise 3 - Model training

In [ ]:
# ── Entraînement de la Régression Logistique ─────────────────────────────────
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

clf = Pipeline([
    ('pre', preprocess),
    ('lr', LogisticRegression(max_iter=1000, random_state=42))
])

clf.fit(X_train, y_train)
print("Modèle entraîné avec succès !")
print("Étapes du pipeline :", [step[0] for step in clf.steps])


## Exercise 4 - Evaluation metrics

- Plot accuracy and comment
- Plot confusion matrix and comment
- Plot precision, recall, F1 and comment


> TODO: comment on the balance between precision and recall.

In [ ]:
# ── Calcul des métriques ───────────────────────────────────────────────────────
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                               f1_score, confusion_matrix, ConfusionMatrixDisplay)

y_pred = clf.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)

print("Accuracy  :", round(acc,  4))
print("Precision :", round(prec, 4))
print("Recall    :", round(rec,  4))
print("F1        :", round(f1,   4))

# ── Graphique des métriques ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(
    ['Accuracy', 'Precision', 'Recall', 'F1'],
    [acc, prec, rec, f1],
    color=['#4C72B0', '#DD8452', '#55A868', '#C44E52']
)
axes[0].set_title('Métriques sur le jeu de test')
axes[0].set_ylabel('Score')
axes[0].set_ylim(0, 1.1)
for bar, val in zip(bars, [acc, prec, rec, f1]):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.02,
                 f'{val:.3f}', ha='center', fontweight='bold')

# Matrice de confusion
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Pas diabète', 'Diabète'])
disp.plot(ax=axes[1], colorbar=False, cmap='Blues')
axes[1].set_title('Matrice de confusion')

plt.tight_layout()
plt.show()

### Commentaires sur les métriques

- **Accuracy = 0.9605** : Très élevée, mais trompeuse ici car le dataset est déséquilibré (91.5 % de non-diabétiques). Le modèle pourrait atteindre ~91.5 % en prédisant toujours 0.

- **Precision = 0.8587** : Quand le modèle prédit "diabète", il a raison dans ~86 % des cas — peu de fausses alarmes.

- **Recall = 0.6400** : Le modèle ne détecte que 64 % des vrais diabétiques. C'est la métrique la plus critique en médecine : rater un diabétique a des conséquences graves.

- **F1 = 0.7334** : Compromis entre précision et rappel. Acceptable mais perfectible.

**Bilan** : En contexte médical, on préfère maximiser le **rappel** (détecter tous les malades), quitte à avoir plus de faux positifs. Un ajustement du seuil de décision (ex. 0.3 au lieu de 0.5) ou une pondération des classes (`class_weight='balanced'`) pourrait améliorer le rappel.


## Exercise 5 - Visualizing the performance of our model

Visualize a 2D decision boundary with accuracy info. Use only two informative features for this plot to keep it 2D. Suggested pair: `HbA1c_level` and `blood_glucose_level` if present. Otherwise pick any two numeric features.


In [ ]:
# ── Frontière de décision sur 2 features ─────────────────────────────────────
feat_x = 'HbA1c_level'       if 'HbA1c_level'         in X.columns else X.select_dtypes(include=['int64','float64']).columns[0]
feat_y = 'blood_glucose_level' if 'blood_glucose_level' in X.columns else X.select_dtypes(include=['int64','float64']).columns[1]

X2_train = X_train[[feat_x, feat_y]].copy()
X2_test  = X_test[[feat_x, feat_y]].copy()

pipe2 = Pipeline([
    ('pre', ColumnTransformer([('num', StandardScaler(), [0, 1])], remainder='drop')),
    ('lr',  LogisticRegression(max_iter=1000, random_state=42))
])
pipe2.fit(X2_train.values, y_train)

# Grille de prédiction
x_min, x_max = X2_train[feat_x].min() - 0.5, X2_train[feat_x].max() + 0.5
y_min, y_max = X2_train[feat_y].min() - 5,   X2_train[feat_y].max() + 5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 400),
                      np.linspace(y_min, y_max, 400))
probs = pipe2.predict_proba(np.c_[xx.ravel(), yy.ravel()])[:, 1].reshape(xx.shape)

acc2 = accuracy_score(y_test, pipe2.predict(X2_test.values))

plt.figure(figsize=(8, 6))
plt.contourf(xx, yy, probs, levels=50, cmap='RdBu_r', alpha=0.6)
plt.colorbar(label='P(diabète)')
cs = plt.contour(xx, yy, probs, levels=[0.5], colors='black', linewidths=2)
plt.clabel(cs, fmt={0.5: 'P = 0.5'}, inline=True, fontsize=11)

# Nuage de points (échantillon de 1000 pour la lisibilité)
sample = X2_test.sample(1000, random_state=42)
y_sample = y_test.loc[sample.index]
scatter = plt.scatter(sample[feat_x], sample[feat_y],
                      c=y_sample, cmap='coolwarm', edgecolors='k',
                      linewidths=0.3, alpha=0.7, s=25)
plt.legend(*scatter.legend_elements(), title="Classe")
plt.xlabel(feat_x, fontsize=12)
plt.ylabel(feat_y, fontsize=12)
plt.title(f'Frontière de décision (2 features) – Accuracy test : {acc2:.3f}', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Accuracy avec seulement {feat_x} et {feat_y} : {acc2:.4f}")


## Exercise 6 - ROC curve

Use the code template provided to plot the ROC curve for your model and compute AUC. You can reuse the fitted `clf` pipeline.

Template summary:
- Get predicted probabilities for the positive class
- Compute fpr and tpr with `roc_curve`
- Plot ROC and print AUC


In [ ]:
# ── Courbe ROC ────────────────────────────────────────────────────────────────
from sklearn import metrics

y_proba = clf.predict_proba(X_test)[:, 1]
fpr, tpr, thresholds = metrics.roc_curve(y_test, y_proba)
auc = metrics.roc_auc_score(y_test, y_proba)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (AUC = {auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--', label='Aléatoire (AUC = 0.500)')
plt.fill_between(fpr, tpr, alpha=0.1, color='darkorange')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('Taux de Faux Positifs (FPR)', fontsize=12)
plt.ylabel('Taux de Vrais Positifs (TPR / Recall)', fontsize=12)
plt.title('Courbe ROC – Régression Logistique', fontsize=13)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

print(f"AUC = {auc:.4f}")

### Interprétation de la courbe ROC et de l'AUC

- **AUC = 0.9625** : Excellent score ! Un AUC proche de 1 signifie que le modèle discrimine très bien les individus diabétiques des non-diabétiques. Un modèle aléatoire aurait AUC = 0.5 (diagonale en pointillés).

- **Lecture de la courbe** : La courbe monte rapidement vers le coin supérieur gauche (TPR ≈ 1, FPR ≈ 0), ce qui montre que le modèle peut atteindre un très bon rappel tout en conservant peu de faux positifs.

- **Avantage de la ROC vs Accuracy** : La ROC est indépendante du seuil de classification. Elle permet de choisir le seuil optimal selon le contexte médical — par exemple, un point de fonctionnement qui maximise le rappel (détecter tous les diabétiques) au prix d'un FPR plus élevé (plus de fausses alarmes acceptables).

- **Conclusion** : Malgré un rappel modéré (0.64) avec le seuil par défaut de 0.5, l'AUC élevée confirme que le modèle a un excellent **potentiel discriminant**. En abaissant le seuil de décision, on peut améliorer le rappel selon les besoins cliniques.

> TODO: interpret the ROC curve and AUC from results of execution of cell above.